<a href="https://colab.research.google.com/github/e23395/Statistical-Learning-e23395/blob/main/Assignment%207b%3A%20Gaussian%20Mixture%20Model%20Clustering%20as%20Conditional%20Updating.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Q1. Bayesian Estimation of a User Ability Parameter from Item Responses**

###**(i) Visualizing the Mechanics:**

In [1]:
import numpy as np
import plotly.graph_objects as go

# Define the 2PL function
def p_i(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

theta = np.linspace(-4, 4, 200)

fig = go.Figure()

# Curve set 1: a_i = 1.0 with varying b_i (three different values)
fig.add_trace(go.Scatter(x=theta, y=p_i(theta, 1.0, -1.0), name="a=1.0, b=-1.0"))
fig.add_trace(go.Scatter(x=theta, y=p_i(theta, 1.0, 0.0), name="a=1.0, b=0.0"))
fig.add_trace(go.Scatter(x=theta, y=p_i(theta, 1.0, 1.0), name="a=1.0, b=1.0"))

# Curve set 2: a_i = 2.5 with one b_i to show contrast in discrimination
fig.add_trace(go.Scatter(x=theta, y=p_i(theta, 2.5, 0.0), name="a=2.5, b=0.0", line=dict(dash='dash')))

fig.update_layout(title="2PL Model Mechanics", xaxis_title="Theta (Ability)", yaxis_title="P(Y_i = 1 | Theta)")
fig.show()

*   Interpretation:

**Horizontal Shift:** The parameter $b_i$ represents the item difficulty and acts as a location shift. Increasing $b_i$ shifts the probability curve horizontally to the right, meaning a higher latent ability $\theta$ is required to maintain the same probability of success. Specifically, when $\theta = b_i$, $P(Y_i = 1 \mid \theta) = 0.5$.

###**(ii) Sequential Likelihood Contribution:**



Single Likelihood Contribution $L(y_k \mid \theta)$:

$$L(y_k \mid \theta) = [p_k(\theta)]^{y_k} [1 - p_k(\theta)]^{1 - y_k}$$

Substituting the 2PL formula:

$$L(y_k \mid \theta) = \left(\frac{1}{1 + e^{-a_k(\theta - b_k)}}\right)^{y_k} \left(\frac{e^{-a_k(\theta - b_k)}}{1 + e^{-a_k(\theta - b_k)}}\right)^{1 - y_k}$$

Joint Likelihood Function for $\mathbf{y}^{(k)}$ (assuming conditional independence):

$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^k [p_i(\theta)]^{y_i} [1 - p_i(\theta)]^{1 - y_i}$$

###**(iii) Mathematical Formulation of the Running Update**



The recursive relationship for the posterior density at step $k$ up to a proportionality constant is:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)}) \cdot L(y_k \mid \theta)$$

Substituting the single likelihood term:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)}) \cdot [p_k(\theta)]^{y_k} [1 - p_k(\theta)]^{1 - y_k}$$

###**(iv) Dynamic Shifting**

When a user correctly answers ($y_k = 1$) a highly difficult item (large $b_k$), the likelihood contribution is $L(y_k \mid \theta) = p_k(\theta)$.Mathematically, the derivative of the log-likelihood contribution with respect to $\theta$ is:

$$\frac{\partial}{\partial \theta} \ln p_k(\theta) = a_k (1 - p_k(\theta)) > 0$$

Because this derivative is strictly positive and largest for values of $\theta < b_k$, multiplying the prior density by this monotonically increasing function heavily weights the upper region of the ability parameter space. This causes the peak (mode) of the updated posterior distribution to shift significantly to the right relative to the previous step.

###**(v) Tracking Certainty and Sharpness**

*   Large $a_k$ (High Discrimination): The function $p_k(\theta)$ transitions sharply from 0 to 1 around $\theta = b_k$. The likelihood acts like a strict filter. This drastically reduces the posterior variance, making the distribution sharper and causing a rapid shift in the estimate toward the item region.

*   Small $a_k$ (Low Discrimination): The function $p_k(\theta)$ is very flat across $\theta$. The likelihood contribution introduces minimal new information, leaving the variance virtually unchanged and the posterior distribution broad.

###**(vi) Numerical Implementation of a Running Grid**

1.   Grid Initialization: Define a fixed fine grid of $M$ evenly spaced points $\theta \in [\theta_{\min}, \theta_{\max}]$, and initialize a vector representing the prior values: $P_0 = [f_{\Theta}^{(0)}(\theta_1), \dots, f_{\Theta}^{(0)}(\theta_M)]$.

2.   Sequential Step (Update & Normalization): When response $y_k$ arrives for an item with parameters $(a_k, b_k)$:
   * Multiply: Calculate the unnormalized posterior vector $U_k$ pointwise:
   $$U_k[m] = P_{k-1}[m] \times [p_k(\theta_m)]^{y_k} [1 - p_k(\theta_m)]^{1 - y_k} \quad \text{for } m = 1, \dots, M$$

   * Normalize: Convert $U_k$ into a proper probability mass function by dividing by the sum of its elements multiplied by the grid spacing $\Delta \theta$:
   $$\text{Area} = \sum_{m=1}^M U_k[m] \cdot \Delta \theta \implies P_k[m] = \frac{U_k[m]}{\text{Area}}$$

   

###**(vii) Evaluating Convergence over the Timeline**

In [6]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# =====================================================================
# CORE ENGINE: ITEM RESPONSE THEORY & BAYESIAN UPDATE FUNCTIONS
# =====================================================================

def compute_2pl_prob(theta_grid, discrimination, difficulty):
    """Calculates success probability under the 2PL model."""
    return 1.0 / (1.0 + np.exp(-discrimination * (theta_grid - difficulty)))

def update_posterior(prior_density, theta_grid, score, discrimination, difficulty):
    """
    Performs a point-wise Bayesian update across a grid and normalizes
    the resulting distribution using the trapezoidal rule.
    """
    success_prob = compute_2pl_prob(theta_grid, discrimination, difficulty)
    # Likelihood application: P(Y=y | theta) = p^y * (1-p)^(1-y)
    likelihood = success_prob if score == 1 else (1.0 - success_prob)

    unnormalized_posterior = prior_density * likelihood
    norm_constant = np.trapezoid(unnormalized_posterior, theta_grid)

    return unnormalized_posterior / norm_constant

# =====================================================================
# PIPELINE 1: MANUAL 4-STEP SEQUENCE TRACKING
# =====================================================================

def run_manual_sequence():
    # 1. Grid & Initial State Setup
    resolution = 500
    param_space = np.linspace(-5, 5, resolution)
    active_density = stats.norm.pdf(param_space, loc=0, scale=1)

    # 2. Dataset Definition
    historical_logs = [
        {"disc": 1.0, "diff": -1.5, "outcome": 1},  # Easy item, Success
        {"disc": 1.5, "diff": 0.5,  "outcome": 1},  # Medium item, Success
        {"disc": 1.2, "diff": 1.5,  "outcome": 0},  # Hard item, Failure
        {"disc": 2.0, "diff": 0.2,  "outcome": 1}   # High discrimination item, Success
    ]

    # 3. Visualization Configuration
    chart = go.Figure()
    chart.add_trace(go.Scatter(
        x=param_space, y=active_density, mode='lines',
        name='Base Prior: N(0,1)', line=dict(dash='dot', width=2, color='#7f8c8d')
    ))

    # 4. Iterative Processing Pipeline
    for step, item in enumerate(historical_logs, start=1):
        active_density = update_posterior(
            prior_density=active_density,
            theta_grid=param_space,
            score=item["outcome"],
            discrimination=item["disc"],
            difficulty=item["diff"]
        )

        status = "Correct" if item["outcome"] == 1 else "Incorrect"
        label = f"Iteration {step}: {status} (a={item['disc']}, b={item['diff']})"
        chart.add_trace(go.Scatter(x=param_space, y=active_density, mode='lines', name=label))

    chart.update_layout(
        title={
            'text': "Evolution of Posterior Latent Ability Density",
            'y': 0.95, 'x': 0.5, 'xanchor': 'center', 'yanchor': 'top'
        },
        xaxis_title="Latent Ability Profile (θ)",
        yaxis_title="Density Amplitude f(θ | Data)",
        template="plotly_white",  # Setting white background template
        hovermode="x unified",
        legend=dict(
            yanchor="top", y=0.95, xanchor="left", x=0.02,
            bgcolor="rgba(255,255,255,0.7)"
        )
    )
    chart.show()

# =====================================================================
# PIPELINE 2: CONVERGENCE TIMELINE SIMULATION (20 ITEMS)
# =====================================================================

def run_convergence_simulation(true_ability=0.75, total_items=20):
    np.random.seed(42)  # Strict environment seed preservation

    # 1. Initialization
    grid_points = np.linspace(-5, 5, 1000)
    density_state = stats.norm.pdf(grid_points, loc=0, scale=1)

    # Generate randomized item vectors ahead of time
    disc_vector = np.random.uniform(0.5, 2.0, size=total_items)
    diff_vector = np.random.normal(0, 1, size=total_items)

    # Metric histories (Index 0 represents the prior parameters)
    history_bayes_mean = [0.0]
    history_map_mode = [0.0]
    timeline_steps = list(range(total_items + 1))

    # 2. Stochastic Simulation & Update Loop
    for k in range(total_items):
        a_k, b_k = disc_vector[k], diff_vector[k]

        # Determine empirical success odds and simulate outcome
        true_success_prob = compute_2pl_prob(true_ability, a_k, b_k)
        observed_outcome = int(np.random.uniform(0, 1) < true_success_prob)

        # Apply sequential filter update
        density_state = update_posterior(density_state, grid_points, observed_outcome, a_k, b_k)

        # Calculate statistical estimators
        expected_value = np.trapezoid(grid_points * density_state, grid_points)
        maximum_mode = grid_points[np.argmax(density_state)]

        history_bayes_mean.append(expected_value)
        history_map_mode.append(maximum_mode)

    # 3. Performance Metrics Rendering
    metrics_chart = go.Figure()

    metrics_chart.add_trace(go.Scatter(
        x=timeline_steps, y=history_bayes_mean, mode='lines+markers',
        name='Posterior Mean (E[θ|Y])', line=dict(color='#2980b9', width=2)
    ))

    metrics_chart.add_trace(go.Scatter(
        x=timeline_steps, y=history_map_mode, mode='lines+markers',
        name='MAP Estimate (Mode)', line=dict(color='#27ae60', width=1.5),
        marker=dict(symbol='diamond')
    ))

    metrics_chart.add_hline(
        y=true_ability, line_dash="dash", line_color="#c0392b", line_width=2,
        annotation_text=f"Target Ability Baseline ({true_ability})",
        annotation_position="top left"
    )

    metrics_chart.update_layout(
        title={
            'text': "Dynamic Estimator Convergence over Item Sequence",
            'y': 0.93, 'x': 0.5, 'xanchor': 'center', 'yanchor': 'top'
        },
        xaxis_title="Chronological Step / Item Count (k)",
        yaxis_title="Estimated Latent Value (θ̂)",
        xaxis=dict(tickmode='linear', tick0=0, dtick=2),
        yaxis=dict(range=[-1.2, 2.2]),
        template="plotly_white",  # Setting white background template
        hovermode="x unified",
        legend=dict(yanchor="top", y=0.15, xanchor="left", x=0.02)
    )
    metrics_chart.show()

# =====================================================================
# EXECUTION CONTROLLER
# =====================================================================
if __name__ == "__main__":
    run_manual_sequence()
    run_convergence_simulation()

**Analysis**

*   Distance Behavioral Shift: As the step metric $k$ increases, the tracking error—measured as the absolute distance between the point estimators ($\hat{\theta}_{\text{Bayes}}^{(k)}$, $\hat{\theta}_{\text{MAP}}^{(k)}$) and the objective parameter baseline ($\theta_{\text{true}} = 0.75$)—characteristically decreases and converges. The early fluctuating deviations shrink rapidly within the first 6 to 10 items.

*   Platform Confidence Implication: Mathematically, this behavioral pattern signals that the variance of the running posterior distribution is rapidly contracting. As the item sequence grows, the cumulative data-driven joint likelihood function starts to completely out-scale and dominate the initialization weight of the standard normal prior $\mathcal{N}(0,1)$.

*   Consequently, the platform's statistical uncertainty decreases and its measurement confidence maximizes, concentrating the density curve tightly around the user's authentic latent proficiency space.

#**Q2. Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates**

###**(i) Structural Probability and Properties**

In [7]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# Define domain over [0, 1]
theta_vals = np.linspace(0, 1, 500)

chart1 = go.Figure()

# Plot the three requested parameter distributions
chart1.add_trace(go.Scatter(x=theta_vals, y=stats.beta.pdf(theta_vals, 1, 1), name="Uninformative (α=1, β=1)"))
chart1.add_trace(go.Scatter(x=theta_vals, y=stats.beta.pdf(theta_vals, 2, 8), name="Right-skewed (α=2, β=8)"))
chart1.add_trace(go.Scatter(x=theta_vals, y=stats.beta.pdf(theta_vals, 8, 2), name="Left-skewed (α=8, β=2)"))

chart1.update_layout(
    title="Beta Distribution Profiles",
    xaxis_title="Conversion Rate (θ)",
    yaxis_title="Probability Density",
    template="plotly_white"
)
chart1.show()

**Interpretation:**

The parameters $\alpha$ and $\beta$ act as pseudo-counts for observed clicks and non-clicks, respectively. The center of mass (mean) is governed by $\frac{\alpha}{\alpha + \beta}$.

*   When $\alpha = \beta$, the density is perfectly symmetric.
*   When $\beta > \alpha$, the mass shifts left (skewed right), reflecting a belief favoring lower CTR values.
*   When $\alpha > \beta$, the mass shifts right (skewed left), reflecting a belief favoring higher CTR values.

###**(ii) Sequential Likelihood and Joint History**

*   Single Contribution $L(y_k \mid \theta)$:

$$L(y_k \mid \theta) = \theta^{y_k}(1 - \theta)^{1 - y_k}$$

*   Joint Likelihood Function for $\mathbf{y}^{(k)}$:

$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^k \theta^{y_i}(1 - \theta)^{1 - y_i} = \theta^{\sum_{i=1}^k y_i} (1 - \theta)^{k - \sum_{i=1}^k y_i}$$

###**(iii) Closed-Form Analytical Updates (Conjugacy)**

**Derivation & Proof:**

By Bayes' Theorem, the sequential posterior update satisfies:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)}) \cdot L(y_k \mid \theta)$$

Substituting the Beta density prior from step $k-1$ and the Bernoulli likelihood:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \left[\theta^{\alpha_{k-1} - 1}(1 - \theta)^{\beta_{k-1} - 1}\right] \cdot \left[\theta^{y_k}(1 - \theta)^{1 - y_k}\right]$$

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \theta^{(\alpha_{k-1} + y_k) - 1} (1 - \theta)^{(\beta_{k-1} + 1 - y_k) - 1}$$

Because this perfectly matches the functional kernel of a Beta distribution, the posterior remains conjugate with exact updates:

$$\alpha_k = \alpha_{k-1} + y_k$$$$\beta_k = \beta_{k-1} + (1 - y_k)$$

Posterior Mean:

$$\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)} = \mathbf{y}^{(k)}] = \frac{\alpha_k}{\alpha_k + \beta_k} = \frac{\alpha_{k-1} + y_k}{\alpha_{k-1} + \beta_{k-1} + 1}$$

###**(iv) Dynamic Shifting Mechanics**

An observed click ($y_k = 1$) increments $\alpha_k$, explicitly shifting the peak of the distribution to the right. A non-click ($y_k = 0$) increments $\beta_k$, shifting the peak to the left.

Contrast against Non-Conjugate Models:
In non-conjugate setups like the 2PL IRT model, updates cannot be written algebraically because the product of the prior density and the logistic likelihood has no closed-form solution. This requires numerical grid integration at every sequential step. In the Beta-Binomial conjugate setup, the update reduces to basic, exact additions ($\alpha + 1$ or $\beta + 1$), running instantly without tracking grids or losing numerical precision.

###**(v) Running Point Estimators**



*   Running Posterior Mean ($\hat{\theta}_{\text{Bayes}}^{(k)}$):

$$\hat{\theta}_{\text{Bayes}}^{(k)} = \frac{\alpha_k}{\alpha_k + \beta_k}$$

*   Running Maximum A Posteriori ($\hat{\theta}_{\text{MAP}}^{(k)}$):

$$\hat{\theta}_{\text{MAP}}^{(k)} = \frac{\alpha_k - 1}{\alpha_k + \beta_k - 2} \quad (\text{for } \alpha_k, \beta_k > 1)$$

###**(vi) Performance Tracking and Convergence Analysis**

In [9]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# =====================================================================
# ENGINE COMPONENT: ANALYTICAL BETA-BINOMIAL CALCULATORS
# =====================================================================

def evaluate_point_estimators(alpha_count, beta_count):
    """
    Computes exact posterior expected value (Mean) and the mode (MAP).
    Applies strict edge protections for uniform or edge-heavy configurations.
    """
    # Expected value calculation
    point_mean = alpha_count / (alpha_count + beta_count)

    # Mode calculation with safety guards for boundary conditions
    if alpha_count > 1 and beta_count > 1:
        point_map = (alpha_count - 1) / (alpha_count + beta_count - 2)
    elif alpha_count == beta_count:
        point_map = 0.5
    else:
        point_map = 1.0 if alpha_count > beta_count else 0.0

    return point_mean, point_map


# =====================================================================
# CONVERGENCE SIMULATION ENGINE
# =====================================================================

def run_ctr_updating_system(target_ctr=0.35, total_trials=100):
    # Enforce isolated execution consistency
    np.random.seed(42)

    # 1. State Space Configurations
    resolution = 500
    probability_domain = np.linspace(0, 1, resolution)
    milestone_checkpoints = {0, 1, 2, 5, 10, 30, 50, 100}

    # 2. Hyperparameter Initialization (Uniform Base Prior Condition)
    current_alpha, current_beta = 1.0, 1.0

    # 3. Dynamic Telemetry Storage Initialization
    initial_mean, initial_map = evaluate_point_estimators(current_alpha, current_beta)
    mean_history = [initial_mean]
    map_history = [initial_map]
    chronology_steps = list(range(total_trials + 1))

    # 4. Canvas Configurations (Plotly Canvas Setup)
    density_plot = go.Figure()

    # Plot baseline distribution state (Step 0)
    density_plot.add_trace(go.Scatter(
        x=probability_domain,
        y=stats.beta.pdf(probability_domain, current_alpha, current_beta),
        mode='lines',
        name='Prior State: Uniform(0,1)',
        line=dict(color='#95a5a6', width=2, dash='dash')
    ))

    # =====================================================================
    # ITERATIVE PROCESSING PHASE
    # =====================================================================
    for operation_step in range(1, total_trials + 1):
        # Stochastic observation arrival
        empirical_success = int(np.random.uniform(0, 1) < target_ctr)

        # Conjugate state accumulation
        current_alpha += empirical_success
        current_beta += (1 - empirical_success)

        # Pull system estimates
        step_mean, step_map = evaluate_point_estimators(current_alpha, current_beta)
        mean_history.append(step_mean)
        map_history.append(step_map)

        # Checkpoint Density Visual Capturing
        if operation_step in milestone_checkpoints:
            curve_y = stats.beta.pdf(probability_domain, current_alpha, current_beta)
            event_type = "Click" if empirical_success == 1 else "No-Click"
            label_string = f"T={operation_step} ({event_type}) | α={int(current_alpha)}, β={int(current_beta)}"

            density_plot.add_trace(go.Scatter(
                x=probability_domain,
                y=curve_y,
                mode='lines',
                name=label_string
            ))

    # =====================================================================
    # LAYOUT GENERATION & PLOTTING (WHITE THEME ENFORCED)
    # =====================================================================

    # Finalize Figure 1: Density Movement
    density_plot.add_vline(
        x=target_ctr, line_dash="longdashdot", line_color="#d63031", line_width=2,
        annotation_text=f"Target CTR ({target_ctr})", annotation_position="top right"
    )
    density_plot.update_layout(
        title={'text': "Conjugate Beta Density Progression Timeline", 'x': 0.5, 'y': 0.95, 'xanchor': 'center'},
        xaxis_title="Latent Conversion Rate (θ)",
        yaxis_title="Probability Density Magnitude",
        template="plotly_white",
        hovermode="x unified",
        legend=dict(yanchor="top", y=0.96, xanchor="right", x=0.98, bgcolor="rgba(255,255,255,0.6)")
    )
    density_plot.show()

    # Finalize Figure 2: Convergence Metrics Trace
    metrics_plot = go.Figure()

    metrics_plot.add_hline(
        y=target_ctr, line_dash="dash", line_color="#d63031", line_width=2,
        annotation_text=f"True Baseline Profile ({target_ctr})", annotation_position="bottom right"
    )
    metrics_plot.add_trace(go.Scatter(
        x=chronology_steps, y=mean_history, mode='lines+markers',
        name='Bayesian Expected Mean', line=dict(color='#0984e3', width=2), marker=dict(size=4)
    ))
    metrics_plot.add_trace(go.Scatter(
        x=chronology_steps, y=map_history, mode='lines',
        name='MAP Maximum Mode', line=dict(color='#27ae60', width=1.5, dash='dot')
    ))
    metrics_plot.update_layout(
        title={'text': "Estimator Trajectory Tracking Analysis", 'x': 0.5, 'y': 0.93, 'xanchor': 'center'},
        xaxis_title="Aggregated User Impressions (k)",
        yaxis_title="Estimated Metric Level (θ̂)",
        template="plotly_white",
        hovermode="x unified",
        legend=dict(yanchor="bottom", y=0.06, xanchor="right", x=0.98)
    )
    metrics_plot.show()

# Run the simulation context
if __name__ == "__main__":
    run_ctr_updating_system()

**Analysis & Interpretation**

*   **Distance to True CTR:** As the sampling size $k$ approaches 100, the absolute distance between both estimators ($\hat{\theta}_{\text{Bayes}}^{(k)}$, $\hat{\theta}_{\text{MAP}}^{(k)}$) and the objective parameter baseline ($\theta_{\text{true}} = 0.35$) monotonically decreases and stabilizes around the target value.

*   **Evidence vs. Initial Prior:** This tells us that as empirical data accumulates over time, the data-driven likelihood function increasingly outweighs the initial prior parameters. Even if an uninformative or highly biased initial prior is used, the system asymptotically washes out the initial state, causing the posterior distribution to compress tightly around the true system behavior.

#**Q3. Bayesian Estimations for Structural Health Monitoring via Bounded Grid Updates**

###**(i) Prior Belief Boundaries**

In [13]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

def run_complete_structural_health_monitoring():
    np.random.seed(24)

    true_factor = 0.68
    nominal_stiffness = 50.0
    noise_deviation = 0.15
    total_milestones = 15

    grid_size = 500
    efficiency_axis = np.linspace(0.01, 1.0, grid_size)

    posterior_distribution = stats.beta.pdf(efficiency_axis, a=8, b=1.5)
    posterior_distribution /= np.trapezoid(posterior_distribution, efficiency_axis)

    checkpoint_steps = [0, 1, 2, 5, 10, 15]

    mean_history = [np.trapezoid(efficiency_axis * posterior_distribution, efficiency_axis)]
    map_history = [efficiency_axis[np.argmax(posterior_distribution)]]
    chronological_steps = list(range(total_milestones + 1))

    density_curves_chart = go.Figure()

    density_curves_chart.add_trace(go.Scatter(
        x=efficiency_axis,
        y=posterior_distribution,
        mode='lines',
        name='Prior State: Initial Estimate',
        line=dict(dash='dash', width=2.5, color='#7f8c8d')
    ))

    for current_step in range(1, total_milestones + 1):
        stochastic_disturbance = np.random.normal(0, noise_deviation)
        empirical_reading = (true_factor * nominal_stiffness) * np.exp(stochastic_disturbance)

        expected_stiffness_matrix = efficiency_axis * nominal_stiffness
        likelihood_response = stats.lognorm.pdf(empirical_reading, s=noise_deviation, scale=expected_stiffness_matrix)

        posterior_distribution = posterior_distribution * likelihood_response
        area_under_curve = np.trapezoid(posterior_distribution, efficiency_axis)
        posterior_distribution /= area_under_curve

        current_mean = np.trapezoid(efficiency_axis * posterior_distribution, efficiency_axis)
        current_map = efficiency_axis[np.argmax(posterior_distribution)]

        mean_history.append(current_mean)
        map_history.append(current_map)

        if current_step in checkpoint_steps:
            density_curves_chart.add_trace(go.Scatter(
                x=efficiency_axis,
                y=posterior_distribution,
                mode='lines',
                name=f"Step {current_step}: Observation K={empirical_reading:.2f}",
                line=dict(width=2)
            ))

    density_curves_chart.add_vline(
        x=true_factor,
        line_dash="dot",
        line_color="#e74c3c",
        line_width=2.5,
        annotation_text=f"True Damage Profile ({true_factor})",
        annotation_position="top left"
    )

    density_curves_chart.update_layout(
        title={'text': "Bayesian Density Field Structural Degradation Progression", 'y': 0.95, 'x': 0.5, 'xanchor': 'center'},
        xaxis_title="Remaining Stiffness Factor (θ)",
        yaxis_title="Probability Density Magnitude",
        template="plotly_white",
        hovermode="x unified",
        legend=dict(yanchor="top", y=0.95, xanchor="left", x=0.02, bgcolor="rgba(255,255,255,0.7)")
    )
    density_curves_chart.show()

    convergence_metrics_chart = go.Figure()

    convergence_metrics_chart.add_hline(
        y=true_factor,
        line_dash="dash",
        line_color="#e74c3c",
        line_width=2,
        annotation_text=f"Target Efficiency ({true_factor})",
        annotation_position="bottom right"
    )

    convergence_metrics_chart.add_trace(go.Scatter(
        x=chronological_steps,
        y=mean_history,
        mode='lines+markers',
        name='Bayes Expected Mean',
        line=dict(color='#2980b9', width=2.5)
    ))

    convergence_metrics_chart.add_trace(go.Scatter(
        x=chronological_steps,
        y=map_history,
        mode='lines+markers',
        name='MAP Maximum Mode',
        line=dict(color='#27ae60', width=1.5, dash='dot')
    ))

    convergence_metrics_chart.update_layout(
        title={'text': "Stiffness Parameter Tracking Calibration Convergence", 'y': 0.93, 'x': 0.5, 'xanchor': 'center'},
        xaxis_title="Inspection Milestones Completed (k)",
        yaxis_title="Estimated Attribute Target (θ̂)",
        template="plotly_white",
        hovermode="x unified",
        legend=dict(yanchor="bottom", y=0.05, xanchor="right", x=0.98)
    )
    convergence_metrics_chart.show()

if __name__ == "__main__":
    run_complete_structural_health_monitoring()

*   **Analytical Expectation:**

$$\mathbb{E}[\Theta^{(0)}] = \frac{\alpha}{\alpha + \beta} = \frac{8}{8 + 1.5} = \frac{8}{9.5} \approx 0.8421$$

*   **Engineering Justification:** This asymmetric density concentrates its mass near $\theta = 1.0$ (perfectly pristine condition) while dropping to zero as $\theta \to 0$. This mathematically encodes the sensible engineering assumption that a newly deployed or freshly inspected component is highly likely to be structural intact and healthy, rather than pre-damaged.

###**(ii) Structural Likelihood Formulation**

Given the non-linear relationship $y_k = \theta \cdot K_{\text{nominal}} \cdot e^{\epsilon_k}$ where $\epsilon_k \sim \mathcal{N}(0, \sigma^2)$, we define $\ln(y_k) = \ln(\theta) + \ln(K_{\text{nominal}}) + \epsilon_k$. Thus, $\ln(y_k) \sim \mathcal{N}(\ln(\theta) + \ln(K_{\text{nominal}}), \sigma^2)$.

Applying the standard log-normal PDF change of variables:

*   Single Continuous Measurement Likelihood $L(y_k \mid \theta)$:

$$L(y_k \mid \theta) = \frac{1}{y_k \sigma \sqrt{2\pi}} \exp \left( -\frac{\left(\ln(y_k) - \ln(\theta) - \ln(K_{\text{nominal}})\right)^2}{2\sigma^2} \right)$$

*   Joint Likelihood Function for Running Vector $\mathbf{y}^{(k)}$:

$$L(\mathbf{y}^{(k)} \mid \theta) = \left(\frac{1}{\sigma \sqrt{2\pi}}\right)^k \left(\prod_{i=1}^k \frac{1}{y_i}\right) \exp \left( -\sum_{i=1}^k \frac{\left(\ln(y_i) - \ln(\theta) - \ln(K_{\text{nominal}})\right)^2}{2\sigma^2} \right)$$

###**(iii) Mathematical Formulation of the Non-Conjugate Grid Update**

An exact algebraic closed-form solution for the posterior density does not exist because multiplying the polynomial kernel of the Beta prior, $\theta^{\alpha-1}(1-\theta)^{\beta-1}$, by the transcendental exponent of the log-normal structural likelihood results in a non-standard distribution mathematical form that cannot be analytically integrated to find its normalizing constant.

*   Recursive Updating Proportionality Relation:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)}) \cdot \frac{1}{y_k} \exp \left( -\frac{\left(\ln(y_k) - \ln(\theta) - \ln(K_{\text{nominal}})\right)^2}{2\sigma^2} \right)$$



###**(iv) Running Point Estimators**

Running Posterior Mean ($\hat{\theta}_{\text{Bayes}}^{(k)}$):

$$\hat{\theta}_{\text{Bayes}}^{(k)} = \frac{\int_{0}^{1} \theta \cdot f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \, d\theta}{\int_{0}^{1} f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \, d\theta}$$

Running Maximum A Posteriori ($\hat{\theta}_{\text{MAP}}^{(k)}$):

$$\hat{\theta}_{\text{MAP}}^{(k)} = \arg\max_{\theta \in (0, 1]} f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$$

###**(v) Algorithmic Grid Approximation and Normalization**

1.   Grid Bounding & Setup: Define a discrete uniform vector of size $M$ over the physical bounds: $\theta_m \in [0.01, 1.0]$. Initialize the prior vector $P_0[m] = \text{Beta.pdf}(\theta_m, 8, 1.5)$.

2.   Sequential Filtering Updates: Upon receiving a new continuous noisy sensor measurement reading $y_k$:
 *   Evaluate the exact likelihood value point-wise across all grid nodes: $L_k[m] = L(y_k \mid \theta_m)$.
 *   Calculate the point-wise unnormalized product: $U_k[m] = P_{k-1}[m] \times L_k[m]$.

 3.   Trapezoidal Normalization: Compute the total area under the unnormalized discrete density using the trapezoidal rule:
 $$\text{Area} = \frac{\Delta \theta}{2} \sum_{m=1}^{M-1} \left( U_k[m] + U_k[m+1] \right)$$
 Normalize the grid array to maintain a proper continuous probability function density distribution: $P_k[m] = \frac{U_k[m]}{\text{Area}}$.

###**(vi) Performance Tracking and Degradation Convergence Analysis**

In [14]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

def run_shm_grid_filter():
    # 1. Setup Ground Truth Environmental Constraints
    np.random.seed(42)
    theta_true = 0.68
    n_measurements = 15
    k_nominal = 50.0
    sigma_noise = 0.15

    # Define fine numeric tracking grid
    grid_resolution = 1000
    theta_grid = np.linspace(0.01, 1.0, grid_resolution)

    # 2. State Initialization: Pristine Prior State Beta(8, 1.5)
    posterior_density = stats.beta.pdf(theta_grid, 8, 1.5)

    # Telemetry histories
    running_bayes = [np.trapezoid(theta_grid * posterior_density, theta_grid)]
    running_map = [theta_grid[np.argmax(posterior_density)]]
    milestones = {0, 1, 2, 5, 10, 15}

    # Canvas initialization
    curves_chart = go.Figure()
    curves_chart.add_trace(go.Scatter(x=theta_grid, y=posterior_density, mode='lines', name='Prior State (k=0)'))

    # 3. Dynamic Filtering Processing Loop
    for k in range(1, n_measurements + 1):
        # Generate noisy physics-based sensor stream readings
        epsilon = np.random.normal(0, sigma_noise)
        y_k = theta_true * k_nominal * np.exp(epsilon)

        # Calculate point-wise Log-Normal Likelihood
        mu_log = np.log(theta_grid) + np.log(k_nominal)
        likelihood = (1.0 / (y_k * sigma_noise * np.sqrt(2 * np.pi))) * np.exp(-((np.log(y_k) - mu_log)**2) / (2 * (sigma_noise**2)))

        # Grid density multiplication update
        posterior_density *= likelihood

        # Explicit normalization via numerical integration
        normalization_integral = np.trapezoid(posterior_density, theta_grid)
        posterior_density /= normalization_integral

        # Extract point estimators
        bayes_mean = np.trapezoid(theta_grid * posterior_density, theta_grid)
        map_mode = theta_grid[np.argmax(posterior_density)]

        running_bayes.append(bayes_mean)
        running_map.append(map_mode)

        # Render density status at specified milestone marks
        if k in milestones:
            curves_chart.add_trace(go.Scatter(x=theta_grid, y=posterior_density, mode='lines', name=f'Posterior Step k={k}'))

    # 4. Finalize Figure 1 Layout (Density Curves)
    curves_chart.update_layout(
        title={'text': "Evolution of Posterior Stiffness Estimates Profile", 'x': 0.5},
        xaxis_title="Stiffness Factor (θ)", yaxis_title="Probability Density Magnitude",
        template="plotly_white"
    )
    curves_chart.show()

    # 5. Finalize Figure 2 Layout (Trajectory Tracking Lines)
    timeline_steps = list(range(n_measurements + 1))
    timeline_chart = go.Figure()
    timeline_chart.add_hline(y=theta_true, line_dash="dash", line_color="red", line_width=2, annotation_text=f"True Stiffness ({theta_true})")
    timeline_chart.add_trace(go.Scatter(x=timeline_steps, y=running_bayes, mode='lines+markers', name='Posterior Mean (Bayes)'))
    timeline_chart.add_trace(go.Scatter(x=timeline_steps, y=running_map, mode='lines+markers', name='MAP Estimate (Mode)'))

    timeline_chart.update_layout(
        title={'text': "Trajectory Tracking Convergence Analysis", 'x': 0.5},
        xaxis_title="Sensor Inspection Milestones (k)", yaxis_title="Estimated Stiffness Level (θ̂)",
        template="plotly_white"
    )
    timeline_chart.show()

if __name__ == "__main__":
    run_shm_grid_filter()

**Analytical Breakdown**

*   System Convergence Window: It takes approximately 2 to 3 sensor readings for the non-conjugate system to override the strongly optimistic "healthy" baseline prior $\mathcal{N}(8, 1.5)$ and confidently isolate the true $68\%$ residual stiffness state ($\theta_{\text{true}} = 0.68$).

*   Implications for Safety Thresholds: The rapid narrowing (variance contraction) of the density curves over the timeline means that the engineering platform's uncertainty drops dramatically within a few continuous steps. In critical structural contexts, this behavior ensures that safety thresholds are reliable: it prevents dangerous false negatives (missing structural cracking due to an overly optimistic prior) by allowing noisy real-world data to swiftly correct the system's baseline assumptions.

#**Q4. Gaussian Mixture Clustering as Conditional Updating**

###**(i) Deriving the Marginal Density**

By the law of total probability, the marginal density $p(x_i)$ is obtained by summing the joint distribution $p(x_i, C_i = k)$ over all possible states of the latent cluster random variable $C_i \in \{1, \dots, K\}$:

$$p(x_i) = \sum_{k=1}^K p(x_i, C_i = k) = \sum_{k=1}^K P(C_i = k) \cdot p(x_i \mid C_i = k)$$

Substituting the prior cluster distribution $P(C_i = k) = \phi_k$ and the multivariate Gaussian component density $X_i \mid C_i = k \sim \mathcal{N}(\mu_k, \Sigma_k)$ yields:

$$p(x_i) = \sum_{k=1}^K \phi_k \mathcal{N}(x_i \mid \mu_k, \Sigma_k)$$

**Interpretation:** This density is called a Gaussian mixture density because it represents a convex combination (or blend) of $K$ distinct multivariate Gaussian distributions, where each component is weighted by its respective mixing proportion $\phi_k \ge 0$ satisfying $\sum_{k=1}^K \phi_k = 1$.

###**(ii) Deriving the Posterior Cluster Probability**

Using Bayes' rule, the conditional probability that observation $x_i$ belongs to cluster $k$ is given by the ratio of the joint density of $x_i$ and $C_i = k$ to the marginal density of $x_i$:

$$P(C_i = k \mid X_i = x_i) = \frac{p(x_i \mid C_i = k) P(C_i = k)}{p(x_i)}$$

Substituting the marginal density derived in Task 1 into the denominator gives:

$$P(C_i = k \mid X_i = x_i) = \frac{\phi_k \mathcal{N}(x_i \mid \mu_k, \Sigma_k)}{\sum_{j=1}^K \phi_j \mathcal{N}(x_i \mid \mu_j, \Sigma_j)} = \gamma_{ik}$$

**Interpretation:** The quantity $\gamma_{ik}$ represents the posterior cluster membership probability (or responsibility) because it updates our prior belief $\phi_k$ about cluster membership after observing the empirical continuous evidence vector $x_i$.

###**(iii) One-Hot Encoding of the Latent Cluster Variable**

Let $Z_{ik} \in \{0, 1\}$ be a binary indicator variable. The conditional expectation of a binary indicator variable is simply the probability of that indicator being equal to 1:

$$\mathbb{E}[Z_{ik} \mid X_i = x_i] = 1 \cdot P(Z_{ik} = 1 \mid X_i = x_i) + 0 \cdot P(Z_{ik} = 0 \mid X_i = x_i)$$

$$\mathbb{E}[Z_{ik} \mid X_i = x_i] = P(C_i = k \mid X_i = x_i) = \gamma_{ik}$$

Stacking these scalar components into a vector configuration yields:

$$\mathbb{E}[Z_i \mid X_i = x_i] = \begin{bmatrix} \gamma_{i1} \\ \gamma_{i2} \\ \vdots \\ \gamma_{iK} \end{bmatrix}$$

**Conclusion:** The soft cluster assignment vector in a GMM is exactly equivalent to the conditional mathematical expectation of the latent structural configuration vector $\mathbb{E}[Z_i \mid X_i = x_i]$.

###**(iv) From Soft Assignment to Hard Clustering**

*   Soft Clustering: Assigns a vector of fractional probabilities $\mathbb{E}[Z_i \mid X_i = x_i] = [\gamma_{i1}, \dots, \gamma_{iK}]^T$ expressing degrees of belief across all categories simultaneously. This accounts for structural uncertainty when points lie near cluster boundaries.

*   Hard Clustering: Collapses this probabilistic distribution into a deterministic single-label selection $\widehat{C}_i = \arg\max_{1 \le k \le K} \gamma_{ik}$. It discards boundary uncertainty, forcing each observation into one explicit group.

###**(v) Conditional Expectation of the Observation Given the Cluster**

Since $X_i \mid C_i = k \sim \mathcal{N}(\mu_k, \Sigma_k)$, the conditional expectation is directly equal to the center of that component distribution:

$$\mathbb{E}[X_i \mid C_i = k] = \int x_i \cdot \mathcal{N}(x_i \mid \mu_k, \Sigma_k) \, dx_i = \mu_k$$

Comparison of Conditional Expectations:

*   $\mathbb{E}[Z_i \mid X_i = x_i]$ maps an observed data point to its soft group membership probabilities (moving from data space to latent space).

*   $\mathbb{E}[X_i \mid C_i = k]$ identifies the continuous center location $\mu_k$ of a given cluster within the data space (moving from latent space to data space).

###**(vi) The Complete-Data Likelihood**

Given the complete-data likelihood equation:

$$p(x_1, \dots, x_n, z_1, \dots, z_n) = \prod_{i=1}^n \prod_{k=1}^K \left[ \phi_k \mathcal{N}(x_i \mid \mu_k, \Sigma_k) \right]^{z_{ik}}$$

Taking the natural logarithm maps the product into a sum:

$$\ell_c = \ln p(x_1, \dots, x_n, z_1, \dots, z_n) = \sum_{i=1}^n \sum_{k=1}^K z_{ik} \left[ \ln \phi_k + \ln \mathcal{N}(x_i \mid \mu_k, \Sigma_k) \right]$$

**Why this expression is easy to maximize:**

If the true discrete labels $z_{ik}$ were perfectly known, the optimization problem uncouples across components. Rather than dealing with a logarithm of sums (as in the marginal log-likelihood), the log completely separates the parameters $\phi_k$, $\mu_k$, and $\Sigma_k$ into independent optimization tasks that can be solved directly via closed-form sample estimators.

###**(vii) The EM Interpretation**

In practice, the latent parameters $Z_i$ are unobserved. The Expectation-Maximization (EM) algorithm handles this by taking the conditional expectation of the complete-data log-likelihood with respect to the posterior distribution of the latent variables given the current parameter estimates $\theta^{\text{old}}$:

$$Q = \mathbb{E}_{Z \mid X, \theta^{\text{old}}}[\ell_c] = \sum_{i=1}^n \sum_{k=1}^K \mathbb{E}[Z_{ik} \mid X_i = x_i, \theta^{\text{old}}] \left[ \ln \phi_k + \ln \mathcal{N}(x_i \mid \mu_k, \Sigma_k) \right]$$

Substituting $\mathbb{E}[Z_{ik} \mid X_i = x_i] = \gamma_{ik}$ yields the structural definition of $Q$:

$$Q = \sum_{i=1}^n \sum_{k=1}^K \gamma_{ik} \left[ \ln \phi_k + \ln \mathcal{N}(x_i \mid \mu_k, \Sigma_k) \right]$$

**E-step Interpretation:** The E-step computes the responsibilities $\gamma_{ik}$ using the current parameter estimates. This can be interpreted as a conditional update of cluster membership probabilities, replacing the unknown binary labels $z_{ik}$ with their soft, data-driven fractional posterior expectations $\gamma_{ik}$.

###**(viii) Parameter Updates (The M-Step)**

Maximizing $Q$ with respect to the model parameters yields standard update formulas:

*   Effective Cluster Population Count ($N_k$):
$$N_k = \sum_{i=1}^n \gamma_{ik}$$

*   Mixing Proportion Update ($\phi_k^{\text{new}}$):
$$\phi_k^{\text{new}} = \frac{N_k}{n}$$

*   Mean Update ($\mu_k^{\text{new}}$):
$$\mu_k^{\text{new}} = \frac{1}{N_k} \sum_{i=1}^n \gamma_{ik} x_i$$

*   Covariance Update ($\Sigma_k^{\text{new}}$):
$$\Sigma_k^{\text{new}} = \frac{1}{N_k} \sum_{i=1}^n \gamma_{ik} (x_i - \mu_k^{\text{new}})(x_i - \mu_k^{\text{new}})^T$$

**Role of Responsibility as a Weight:** The posterior responsibility $\gamma_{ik}$ acts as a fractional membership weight. Instead of an observation contributing fully to just one cluster, its influence is split across components proportional to $\gamma_{ik}$, balancing how much it pulls each cluster's mean and covariance toward its coordinates.

###**(ix) Interpretation**

Gaussian Mixture Model (GMM) clustering can be viewed as an iterative process of conditional updating. The mixture weight $\phi_k$ represents the initial prior probability of selecting cluster $k$ before reviewing any empirical data coordinates. When an observation $x_i$ is introduced, its compatibility with cluster $k$ is evaluated through the multivariate Gaussian density $\mathcal{N}(x_i \mid \mu_k, \Sigma_k)$. Applying Bayes' rule combines this prior and likelihood to generate the responsibility $\gamma_{ik}$, which functions as the updated posterior probability of cluster membership given the data.Collectively, these scalar responsibilities form the soft assignment vector $\mathbb{E}[Z_i \mid X_i = x_i]$. During the M-step, these posterior probabilities serve as conditional weights to re-estimate the cluster parameters ($\phi_k$, $\mu_k$, $\Sigma_k$). This confirms that GMM clustering is a fully probabilistic approach grounded in the conditional expectations of latent membership variables.

###**(x) Computational Simulation and Out-of-Sample Validation**

In [15]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

class GMMFinancialSegmenter:
    def __init__(self, target_clusters=3, seed_value=42):
        self.target_clusters = target_clusters
        self.seed_value = seed_value
        self.feature_normalizer = StandardScaler()
        self.mixture_model = GaussianMixture(
            n_components=self.target_clusters,
            covariance_type="full",
            random_state=self.seed_value
        )
        self.training_inputs = None
        self.validation_inputs = None

    def prepare_data(self, dataset, column_targets, holdout_ratio=0.2):
        cleaned_matrix = dataset[column_targets].dropna().values

        normalized_matrix = self.feature_normalizer.fit_transform(cleaned_matrix)

        self.training_inputs, self.validation_inputs = train_test_split(
            normalized_matrix,
            test_size=holdout_ratio,
            random_state=self.seed_value
        )
        return self.training_inputs, self.validation_inputs

    def fit_model(self):
        self.mixture_model.fit(self.training_inputs)
        print(">> Core GMM Parameter Estimation Finalized.")
        print(f">> Convergence Achieved: {self.mixture_model.converged_}")
        print(f">> Operational Iterations: {self.mixture_model.n_iter_}")

    def evaluate_performance(self):
        mean_log_likelihood = self.mixture_model.score(self.validation_inputs)
        print(f"\n>> Out-of-Sample Performance Metric:")
        print(f">> Mean Log-Likelihood: {mean_log_likelihood:.4f}")
        return mean_log_likelihood

    def generate_density_heatmap(self, target_labels):
        unscaled_training = self.feature_normalizer.inverse_transform(self.training_inputs)

        density_plot = px.density_heatmap(
            x=unscaled_training[:, 0],
            y=unscaled_training[:, 1],
            labels={"x": target_labels[0], "y": target_labels[1]},
            title="Empirical Feature Spaces Density Distribution Profile",
            marginal_x="histogram",
            marginal_y="histogram"
        )

        density_plot.update_traces(colorscale="Viridis", selector=dict(type='histogram2d'))
        density_plot.update_layout(template="plotly_white", width=900, height=700)
        density_plot.show()

    def _evaluate_probability_mesh(self, target_matrix, structural_steps=200):
        horizontal_min, horizontal_max = target_matrix[:, 0].min() - 0.5, target_matrix[:, 0].max() + 0.5
        vertical_min, vertical_max = target_matrix[:, 1].min() - 0.5, target_matrix[:, 1].max() + 0.5

        horizontal_grid, vertical_grid = np.meshgrid(
            np.linspace(horizontal_min, horizontal_max, structural_steps),
            np.linspace(vertical_min, vertical_max, structural_steps)
        )

        flattened_mesh = np.c_[horizontal_grid.ravel(), vertical_grid.ravel()]
        posterior_probabilities = self.mixture_model.predict_proba(flattened_mesh)
        peak_responsibility_map = posterior_probabilities.max(axis=1).reshape(horizontal_grid.shape)

        unscaled_mesh_points = self.feature_normalizer.inverse_transform(flattened_mesh)
        unscaled_horizontal = unscaled_mesh_points[:, 0].reshape(horizontal_grid.shape)
        unscaled_vertical = unscaled_mesh_points[:, 1].reshape(vertical_grid.shape)

        return unscaled_horizontal, unscaled_vertical, peak_responsibility_map

    def render_assignment_boundaries(self, observational_matrix, focus_labels, system_phase="Training"):
        unscaled_horizontal, unscaled_vertical, peak_responsibility_map = self._evaluate_probability_mesh(observational_matrix)
        deterministic_assignments = self.mixture_model.predict(observational_matrix)
        unscaled_observations = self.feature_normalizer.inverse_transform(observational_matrix)

        assignment_canvas = go.Figure()

        assignment_canvas.add_trace(go.Contour(
            x=unscaled_horizontal[0, :],
            y=unscaled_vertical[:, 0],
            z=peak_responsibility_map,
            colorscale="Cividis",
            contours_coloring="heatmap",
            name="Max Cluster Target Responsibility (γ_ik)",
            hoverinfo="skip",
            opacity=0.6
        ))

        for cluster_idx in range(self.target_clusters):
            membership_mask = deterministic_assignments == cluster_idx
            assignment_canvas.add_trace(go.Scatter(
                x=unscaled_observations[membership_mask, 0],
                y=unscaled_observations[membership_mask, 1],
                mode="markers",
                name=f"{system_phase} Component Segment {cluster_idx + 1}",
                marker=dict(size=6, line=dict(width=1, color="Black"))
            ))

        assignment_canvas.update_layout(
            title=f"GMM Continuous Soft-Assignment Boundary Tracking ({system_phase} Framework)",
            xaxis_title=focus_labels[0],
            yaxis_title=focus_labels[1],
            template="plotly_white",
            width=900,
            height=650
        )
        assignment_canvas.show()

if __name__ == "__main__":
    np.random.seed(42)
    mock_financial_data = pd.DataFrame({
        "PURCHASES": np.hstack([
            np.random.exponential(400, 400),
            np.random.normal(2500, 600, 300),
            np.random.normal(6000, 1200, 100),
        ]),
        "CREDIT_LIMIT": np.hstack([
            np.random.normal(2000, 800, 400),
            np.random.normal(7000, 1500, 300),
            np.random.normal(12000, 2000, 100),
        ]),
    })

    selected_features = ["PURCHASES", "CREDIT_LIMIT"]

    segmenter = GMMFinancialSegmenter(target_clusters=3)
    train_data, test_data = segmenter.prepare_data(mock_financial_data, selected_features)

    segmenter.fit_model()
    segmenter.evaluate_performance()

    segmenter.generate_density_heatmap(selected_features)
    segmenter.render_assignment_boundaries(train_data, selected_features, system_phase="Training")
    segmenter.render_assignment_boundaries(test_data, selected_features, system_phase="Testing")

>> Core GMM Parameter Estimation Finalized.
>> Convergence Achieved: True
>> Operational Iterations: 4

>> Out-of-Sample Performance Metric:
>> Mean Log-Likelihood: -1.1576
